What is Machine Learning?
===
Machine learning has evolved into a powerful tool that benefits a wide variety of tasks. Although these tasks may appear diverse, they often share key characteristics — and mathematics provides an ideal framework for describing them. Let us outline a general machine learning problem:

> We are interested in describing ("learning") some function \( f: X \longrightarrow Y \), i.e. \( f(x) = y \), in an explanatory manner, meaning that we have some data at hand which we want to use.

Now, this is fairly abstract, so let us look at some concrete examples.

Task
===
Consider the following pairs of data $(x,y)$:
- $f(0)=0$
- $f(1)=1$
- $f(2)=4$
- $f(3)=9$

What would be a good choice of function $f: \mathbb R \longrightarrow \mathbb R$ to approximately (or even accurately?) describe this data?

In [1]:
# Setup and import needed libraries

# Utility
import matplotlib.pyplot as plt
import numpy as np

import urllib
from PIL import Image

# Machine learning
import torch
import torchvision
from torchvision import transforms

Image Segmentation
---
Here, the input to our function $f$ is an image $x \in \mathbb{R}^{\text{width} \times \text{height} \times \text{channels}}$, and the output $y = f(x)$ is also an image $y \in \mathbb{R}^{\text{width} \times \text{height} \times \text{channels}}$ which segments the various objects.

Below is an example of a pre-trained model for image segmentation. The model and example code are publicly available at:

[https://pytorch.org/hub/pytorch_vision_fcn_resnet101/](https://pytorch.org/hub/pytorch_vision_fcn_resnet101/)

In [2]:
# Download the model
model = torch.hub.load('pytorch/vision:v0.10.0', 'fcn_resnet50', pretrained=True)
model.eval()

# Download an example image to segment
#url = 'https://github.com/pytorch/hub/raw/master/images/dog.jpg'
#image_name = 'dog.jpg' 

# Other possible image from website:
url = 'https://github.com/pytorch/hub/raw/master/images/deeplab1.png'
image_name = 'deeplab1.png'

try: urllib.URLopener().retrieve(url, filename)
except: urllib.request.urlretrieve(url, filename)

# Make some pre-processing
input_image = Image.open(filename)
input_image = input_image.convert("RGB")
preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

input_tensor = preprocess(input_image)
input_batch = input_tensor.unsqueeze(0) # create a mini-batch as expected by the model

# Switch to GPU if possible
use_gpu = True if torch.cuda.is_available() else False

if use_gpu:
    input_batch = input_batch.to('cuda')
    model.to('cuda')

##################################
# The "Magic"
##################################

# Segment the image
with torch.no_grad():
    output = model(input_batch)['out'][0]
output_predictions = output.argmax(0)

# Display the image
# create a color pallette, selecting a color for each class
palette = torch.tensor([2 ** 25 - 1, 2 ** 15 - 1, 2 ** 21 - 1])
colors = torch.as_tensor([i for i in range(21)])[:, None] * palette
colors = (colors % 255).numpy().astype("uint8")

# plot the semantic segmentation predictions of 21 classes in each color
r = Image.fromarray(output_predictions.byte().cpu().numpy()).resize(input_image.size)
r.putpalette(colors)

fig, axes = plt.subplots(ncols=2, figsize=(10, 5))  # 2 columns

axes[0].imshow(input_image)
axes[0].set_title('Original Image $x$')
axes[0].axis('off')  # optional

axes[1].imshow(r)
axes[1].set_title('Segmented Image $y=f(x)$')
axes[1].axis('off')  # optional

plt.tight_layout()
plt.show()

Using cache found in /home/hoeflerm/.cache/torch/hub/pytorch_vision_v0.10.0
/home/hoeflerm/.local/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/hoeflerm/.local/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FCN_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1`. You can also use `weights=FCN_ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


NameError: name 'filename' is not defined

Crash Course on Image Representation
---
We want to investigate how an image is represented numerically. In principle, an image is just a table where the cells represent individual pixels.

If the image is **grayscale**, each cell stores one value — typically between 0 and 1, or between 0 and 255 — indicating the amount of "whiteness".

If it is an **RGB (Red-Green-Blue)** image, then each cell stores **three values** corresponding to the amounts of "redness", "greenness", and "blueness".

Let us have a closer look.

In [ ]:
# Make the image representable as a "table"
numpy_image = np.array(input_image.convert('RGB'))

print(f'The original image $x$ has {numpy_image.shape[:2]} pixels and {numpy_image.shape[2]} color channels since it is an RGB image.')

# Let us display the first 10x10 pixel at the top right corner
x, y = 550, 730
width, height = 10, 10

# Select a small window of the picture
view = numpy_image[x:x+width,y:y+height,:]

fig, axes = plt.subplots(ncols=4, figsize=(10, 3))

axes[0].imshow(view)
axes[0].set_title('Color image')
axes[0].axis('off')

axes[1].imshow(view[:,:,0], cmap='grey')
axes[1].set_title('Red channel')
axes[1].axis('off')

axes[2].imshow(view[:,:,1], cmap='grey')
axes[2].set_title('Green channel')
axes[2].axis('off')

axes[3].imshow(view[:,:,2], cmap='grey')
axes[3].set_title('Blue channel')
axes[3].axis('off')

fig.suptitle(f"{width}x{height} pixel section of $x$")
plt.tight_layout()
plt.show()

Image Generation
---
Another powerful application of machine learning is learning how to generate new images. Conceptually, this is a bit more involved, so we'll just sketch the idea here.

Now, $f(x) = y$ still takes as input an image $x \in \mathbb{R}^{\text{width} \times \text{height} \times \text{channels}}$, but the output is now a **single number** $y \in [0, 1]$ representing a probability. The idea is:

- If $x$ is a realistic image, then $f(x) \approx 1$.
- If $x$ is just random noise, then $f(x) \approx 0$.

To actually **generate new images**, we need two ingredients:

- Learn our function $f$ from the data we have.
- Search for images $x$ such that $f(x)$ is high (i.e., close to 1) — this is a **maximization problem**!

The following model and example code are available at:

[https://pytorch.org/hub/facebookresearch_pytorch-gan-zoo_dcgan/](https://pytorch.org/hub/facebookresearch_pytorch-gan-zoo_dcgan/)

In [ ]:
# Download the model
model = torch.hub.load('facebookresearch/pytorch_GAN_zoo:hub', 'DCGAN', pretrained=True, useGPU=use_gpu)

num_images = 8
noise, _ = model.buildNoiseData(num_images)
with torch.no_grad():
    generated_images = model.test(noise)

# Post-processing
grid = torchvision.utils.make_grid(generated_images).permute(1, 2, 0).cpu().numpy()
grid = (grid - grid.min()) / (grid.max() - grid.min())

fig, ax = plt.subplots(figsize=(14, 8))
ax.imshow(grid)
ax.axis('off')

Image Classification
---
Here, $f(x) = y$ also takes as input an image $x$ and outputs a probability $y \in [0, 1]$. However, $y$ now indicates whether the image belongs to a certain class.

In the example later on (*cats_and_dogs.ipynb*), $y$ will be used to indicate whether $x$ shows a **cat** or a **dog**.
